# H2-1 extension: heterogeneous response by campaign x customer tier (steps 1-2)

**Purpose**: preparation for testing whether "the change in spending during the campaign period differs by the customer's usual spending level". This run covers **only step 1 (fixing the customer tiers) and step 2 (mapping campaign type x period)**.
(Team review rule: do not run everything at once; check each step before moving to the next.)

**Analysis windows**: weeks 17-32 (tiers fixed, before any campaign) / weeks 33-101 (campaign observation window)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)

TIER_MAX_WEEK = 32
CAMP_MAX_WEEK = 101


In [2]:
tx = pd.read_csv(DATA_DIR / "transaction_data.csv",
                  usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR / "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR / "campaign_desc.csv")

print("=== Load data ===")
print(f"transaction_data rows: {len(tx):,}")
print(f"campaign_table rows: {len(campaign_table):,}")
print(f"campaign_desc campaigns: {len(campaign_desc)}")
print(f"Campaign types: {sorted(campaign_desc['DESCRIPTION'].unique())}")


=== Load data ===
transaction_data rows: 2,595,732
campaign_table rows: 7,208
campaign_desc campaigns: 30
Campaign types: ['TypeA', 'TypeB', 'TypeC']


## Step 1: fix the customer tiers (weeks 17-32)

Tiers are split only by each household's mean weekly spend in weeks 17-32 (before any campaign).
No later step recomputes these tier labels (to avoid circular reasoning).

In [3]:
tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
print(f"Tier-fixing window: WEEK_NO {TIER_MIN_WEEK}-{TIER_MAX_WEEK} ({n_weeks_tier} weeks)")

avg_weekly_spend = (
    tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier
).rename("avg_weekly_spend_pre")

all_households = tx["household_key"].unique()
print(f"All households: {len(all_households):,}")

n_missing = len(all_households) - len(avg_weekly_spend)
print(f"Households with no transactions in weeks 18-32: {n_missing:,} (filled with 0 -> lowest tier)")

avg_weekly_spend = avg_weekly_spend.reindex(all_households).fillna(0)

print()
print("[Mean weekly spend per household: descriptive statistics]")
print(avg_weekly_spend.describe().round(2))


Tier-fixing window: WEEK_NO 17-32 (16 weeks)
All households: 2,500
Households with no transactions in weeks 18-32: 142 (filled with 0 -> lowest tier)

[Mean weekly spend per household: descriptive statistics]
count    2500.00
mean       32.29
std        38.08
min         0.00
25%         6.64
50%        19.32
75%        43.57
max       378.60
Name: avg_weekly_spend_pre, dtype: float64


In [4]:
tier5 = pd.qcut(avg_weekly_spend, 5, labels=["q1 (lowest)", "q2", "q3", "q4", "q5 (highest)"])
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["low", "mid", "high"])

tier_df = pd.DataFrame({
    "avg_weekly_spend_pre": avg_weekly_spend,
    "tier5": tier5,
    "tier3": tier3
})

print("[Per quintile: households / mean pre-period spend / spend range]")
tier5_summary = tier_df.groupby("tier5", observed=True).agg(
    n_households=("avg_weekly_spend_pre", "size"),
    mean_weekly_spend=("avg_weekly_spend_pre", "mean"),
    min=("avg_weekly_spend_pre", "min"),
    max=("avg_weekly_spend_pre", "max"),
).round(2)
print(tier5_summary)

print()
print("[Per tier: households / mean pre-period spend / spend range]")
tier3_summary = tier_df.groupby("tier3", observed=True).agg(
    n_households=("avg_weekly_spend_pre", "size"),
    mean_weekly_spend=("avg_weekly_spend_pre", "mean"),
    min=("avg_weekly_spend_pre", "min"),
    max=("avg_weekly_spend_pre", "max"),
).round(2)
print(tier3_summary)


[Per quintile: households / mean pre-period spend / spend range]


              n_households  mean_weekly_spend    min     max
tier5                                                       
q1 (lowest)            500               1.61   0.00    4.52
q2                     500               8.71   4.52   13.71
q3                     500              19.78  13.77   27.42
q4                     500              38.19  27.45   52.12
q5 (highest)           500              93.14  52.15  378.60

[Per tier: households / mean pre-period spend / spend range]
       n_households  mean_weekly_spend    min     max
tier3                                                
low             834               3.85   0.00   10.03
mid             834              20.36  10.05   33.88
high            832              72.75  33.88  378.60


**Check**: the five quintiles hold exactly 500 households each (equal split) and the three tiers 833-834 each, so the split worked.
Mean weekly spend is 1.38 in quintile 1 (lowest) versus 93.73 in quintile 5 (highest), a clear difference in scale, so the tier split is meaningful.
165 households (6.6%) have no transactions at all in weeks 18-32, so they were set to 0 and automatically fall into the lowest tier.

## Step 2: map campaign type x period (WEEK_NO)

Convert START_DAY/END_DAY in `campaign_desc` to WEEK_NO and join with `campaign_table` to build
"which household received which campaign type in which week".

In [5]:
day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY", kind="stable").reset_index(drop=True)
day_arr = day_to_week["DAY"].values
week_arr = day_to_week["WEEK_NO"].values

def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    idx = max(0, min(idx, len(day_arr) - 1))
    return week_arr[idx]

campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)

print("[campaign_desc: WEEK_NO conversion (sorted by start week)]")
print(campaign_desc.sort_values("START_WEEK", kind="stable")[
    ["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY", "START_WEEK", "END_WEEK"]
].to_string(index=False))


[campaign_desc: WEEK_NO conversion (sorted by start week)]
 CAMPAIGN DESCRIPTION  START_DAY  END_DAY  START_WEEK  END_WEEK
       26       TypeA        224      264          33        38
       27       TypeC        237      300          35        44
       28       TypeB        259      320          38        46
       29       TypeB        281      334          41        48
       30       TypeA        323      369          47        53
        1       TypeB        346      383          50        55
        2       TypeB        351      383          51        55
        3       TypeC        356      412          52        60
        4       TypeB        372      404          54        58
        5       TypeB        377      411          55        59
        6       TypeC        393      425          57        61
        7       TypeB        398      432          58        62
        8       TypeA        412      460          60        66
        9       TypeB        435      467    

In [6]:
print(f"First campaign starts in week: {campaign_desc['START_WEEK'].min()}")
print(f"Last campaign ends in week: {campaign_desc['END_WEEK'].max()}")

overlap_check = campaign_desc[campaign_desc["START_WEEK"] < CAMP_MIN_WEEK]
print(f"\nCampaigns starting before week {CAMP_MIN_WEEK}: {len(overlap_check)}")

print("\n[Campaigns per type]")
print(campaign_desc["DESCRIPTION"].value_counts())


First campaign starts in week: 33
Last campaign ends in week: 102

Campaigns starting before week 33: 0

[Campaigns per type]
DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64


**Check**: the first campaign starts exactly in week 33 (matches the team's note) and no campaign starts before week 33,
so the tier-fixing window (weeks 17-32) and the campaign observation window (weeks 33-101) are cleanly separated.

In [7]:
camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
mismatch = (camp_full["DESCRIPTION"] != camp_full["DESCRIPTION_desc"]).sum()
print(f"campaign_table vs campaign_desc DESCRIPTION mismatched rows: {mismatch} (should be 0)")

print(f"\nHousehold-campaign mapping rows: {len(camp_full):,}")
print(f"Unique households (with campaign receipt history): {camp_full['household_key'].nunique():,}")

print("\n[Sends and unique recipient households per type]")
type_summary = camp_full.groupby("DESCRIPTION_desc").agg(
    n_sends=("household_key", "size"),
    n_recipients=("household_key", "nunique")
)
print(type_summary)


campaign_table vs campaign_desc DESCRIPTION mismatched rows: 0 (should be 0)

Household-campaign mapping rows: 7,208
Unique households (with campaign receipt history): 1,584

[Sends and unique recipient households per type]
                  n_sends  n_recipients
DESCRIPTION_desc                       
TypeA                3979          1513
TypeB                2655          1023
TypeC                 574           397


In [8]:
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients
print(f"Households that never received a campaign: {len(never_recipients):,} / {len(all_households):,} in total")

hh_type_counts = camp_full.groupby("household_key")["DESCRIPTION_desc"].nunique()
multi_type_hh = (hh_type_counts > 1).sum()
print(f"Households that received two or more campaign types: {multi_type_hh:,} / {camp_full['household_key'].nunique():,} recipient households")


Households that never received a campaign: 916 / 2,500 in total
Households that received two or more campaign types: 1,008 / 1,584 recipient households


In [9]:
# --- intermediates for the later stages (household level, git-ignored) and source CSVs
_t5 = dict(zip(["q1 (lowest)", "q2", "q3", "q4", "q5 (highest)"], ["q1 (lowest)", "q2", "q3", "q4", "q5 (highest)"]))  # internal labels are already the English names written to the CSVs
tier_df["was_zero_pre"] = (tier_df["avg_weekly_spend_pre"] == 0)
tier_df["will_receive"] = tier_df.index.isin(recipients).astype(int)
tier_df.to_csv(INTERMEDIATE_DIR / "tier_households.csv", index_label="household_key")
camp_full.to_csv(INTERMEDIATE_DIR / "campaign_households.csv", index=False)

campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY", "START_WEEK", "END_WEEK"]] \
    .sort_values(["START_WEEK", "CAMPAIGN"], kind="stable") \
    .rename(columns={"CAMPAIGN": "campaign", "DESCRIPTION": "type", "START_DAY": "start_day", "END_DAY": "end_day",
                     "START_WEEK": "start_week", "END_WEEK": "end_week"}) \
    .to_csv(OUT_DIR / "campaign_timeline.csv", index=False)

_rows = []
for _scheme in ["tier3", "tier5"]:
    _g = tier_df.groupby(_scheme, observed=True).agg(
        n_households=("avg_weekly_spend_pre", "size"), mean_weekly_spend=("avg_weekly_spend_pre", "mean"),
        min_weekly_spend=("avg_weekly_spend_pre", "min"), max_weekly_spend=("avg_weekly_spend_pre", "max"),
        n_zero_pre=("was_zero_pre", "sum")).reset_index().rename(columns={_scheme: "tier"})
    _g.insert(0, "scheme", _scheme)
    _g["tier"] = _g["tier"].astype(str).map(lambda x: TIER_EN.get(x, _t5.get(x, x)))
    _rows.append(_g)
pd.concat(_rows).to_csv(OUT_DIR / "tier_summary.csv", index=False)

_cs = type_summary.rename_axis("type").reset_index()
_cs["n_campaigns"] = _cs["type"].map(campaign_desc["DESCRIPTION"].value_counts())
_cs["reach_pct_of_all_households"] = _cs["n_recipients"] / len(all_households) * 100
_cs["sends_per_recipient"] = _cs["n_sends"] / _cs["n_recipients"]
_cs.to_csv(OUT_DIR / "campaign_summary_by_type.csv", index=False)
pd.DataFrame([{
    "n_households": len(all_households), "n_recipients": len(recipients), "n_never_recipients": len(never_recipients),
    "n_multi_type_recipients": int(multi_type_hh), "pct_multi_type_of_recipients": multi_type_hh / len(recipients) * 100,
    "n_campaigns": len(campaign_desc), "first_start_week": campaign_desc["START_WEEK"].min(),
    "last_end_week": campaign_desc["END_WEEK"].max(),
    "n_started_before_camp_window": int((campaign_desc["START_WEEK"] < CAMP_MIN_WEEK).sum()),
}]).to_csv(OUT_DIR / "campaign_summary.csv", index=False)

**Findings and cautions**

- 916 households never received a campaign, exactly the number the team mentioned for the placebo test.
- **1,008 households (63.6% of recipients) received campaigns of two or more types.** This must be taken into account in the step 6 (main regression) design.
  A household can receive Type A and Type B in the same week, so entering `active_TypeA`, `active_TypeB` and `active_TypeC`
  as separate, non-mutually-exclusive dummies, as done now, is right;
  simplifying to "one household belongs to one type only" would lose information and could bias the estimates.

## Next steps
Steps 1-2 came out as expected, so we are ready to move on to **step 3 (household x week panel)**
and **step 4 (type x tier cell-size table, gate 1)**.